In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv('../data/credito_csv.zip')
X = df.drop('Approved', axis=1)
y = df['Approved']

X = pd.get_dummies(X, drop_first=True)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", mode="min", patience=20)

def treinar_e_avaliar_modelo(arquitetura, ativacao, nome_modelo):
    print(f"\n{'='*10} {nome_modelo} {'='*10}")
    print(f"Arquitetura: {arquitetura} neurônios | Ativação: {ativacao}")
    
    model = Sequential()
    
    model.add(Dense(units=arquitetura[0], activation=ativacao, input_shape=(X_train.shape[1],)))
    
    for units in arquitetura[1:]:
        model.add(Dense(units=units, activation=ativacao))
        
    model.add(Dense(units=1, activation='sigmoid'))
    
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    model.fit(x=X_train, y=y_train, epochs=150, 
              validation_data=(X_val, y_val), 
              callbacks=[early_stop], verbose=0)
    
    preds = (model.predict(X_test) > 0.5).astype("int32")
    
    print(classification_report(y_test, preds))
    
    recall = recall_score(y_test, preds)
    return model, recall

modelo_1, rec_1 = treinar_e_avaliar_modelo([32, 16], 'relu', "Modelo 1")

modelo_2, rec_2 = treinar_e_avaliar_modelo([128, 64, 32], 'relu', "Modelo 2")

modelo_3, rec_3 = treinar_e_avaliar_modelo([64, 32], 'tanh', "Modelo 3")


========== Modelo 1 ==========
Arquitetura: [32, 16] neurônios | Ativação: relu


/home/codespace/.local/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-06-07 22:36:26.758326: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
              precision    recall  f1-score   support

           0       0.82      0.87      0.84        52
           1       0.86      0.81      0.83        52

    accuracy                           0.84       104
   macro avg       0.84      0.84      0.84       104
weighted avg       0.84      0.84      0.84       104


========== Modelo 2 ==========
Arquitetura: [128, 64, 32] neurônios | Ativação: relu


/home/codespace/.local/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
              precision    recall  f1-score   support

           0       0.79      0.81      0.80        52
           1       0.80      0.79      0.80        52

    accuracy                           0.80       104
   macro avg       0.80      0.80      0.80       104
weighted avg       0.80      0.80      0.80       104


========== Modelo 3 ==========
Arquitetura: [64, 32] neurônios | Ativação: tanh


/home/codespace/.local/lib/python3.12/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/stepWARNING:tensorflow:6 out of the last 12 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x77af85218e00> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
              precision    recall  f1-score   support

           0       0.79      0.85      0.81        52
           1       0.83      0.77      0.80        52

    accuracy                          

A rede com o maior recall foi o Modelo 1. Obteve uma Acurácia de 0.84, Precisão de 0.86 e Recall de 0.81 para a classe 1 (Aprovado). Os parâmetros utilizados foram: 2 camadas escondidas, com [32, 16] neurônios, utilizando a função de ativação 'relu' e o otimizador Adam.

In [ ]:
melhor_modelo = modelo_1
melhor_modelo.save_weights("pesos_melhor_modelo_credito.weights.h5")

print("Pesos do Modelo 1 salvos com sucesso!")

Pesos do Modelo 1 salvos com sucesso!


In [ ]:
X_amostra = X_test[:20]
y_real_amostra = y_test[:20].values

previsoes_amostra = (melhor_modelo.predict(X_amostra) > 0.5).astype("int32").flatten()

df_resultados = pd.DataFrame({
    'Classe Real': y_real_amostra,
    'Previsão do Modelo': previsoes_amostra
})

df_resultados['Acerto'] = df_resultados['Classe Real'] == df_resultados['Previsão do Modelo']

display(df_resultados)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


,Classe Real,Previsão do Modelo,Acerto
0,0,0,True
1,1,0,False
2,0,0,True
3,0,0,True
4,1,0,False
5,1,0,False
6,0,0,True
7,1,1,True
8,0,0,True
9,0,0,True


O Recall mede a proporção de instâncias positivas reais que o modelo identificou corretamente. Neste contexto de crédito, um alto recall (como o do Modelo 1) indica que a rede produziu poucos Falsos Negativos  ou seja, ela raramente negou crédito para um cliente que, na realidade, merecia a aprovação. Observando a tabela de amostra gerada acima, percebe-se que as previsões do modelo (1) estão fortemente alinhadas com as classes reais (1), justificando a alta taxa de acerto nos verdadeiros positivos.